# Build Step 4 — Validation & Routing Worker (Direct-to-Volume Flow)

**Framework:** File Validation & Routing (`fvr`) • **Deliverable:** the event-driven worker that completes the pipeline — **identify → validate → route → move bytes → evidence + events** — reading staged files from the UC staging volume and landing them in intake or quarantine • **Checkpoint:** completes the build side of **R2** (dry-run matrix + load test) together with Step 3

## 1. Where the worker sits (and the rules it lives by)

```text
Mongo event_log  (FILE_ARRIVED, status RECEIVED)
        │  ① claim (atomic RECEIVED→PROCESSING — competing consumers, N pods)
        ▼
   identify  ② file_type via file_types.identification.patterns (explicit > filename > header)
        ▼
   validate  ③ rule engine over validation_rules + schema_versions (HEADER/CONTENT category)
              → validation_results rows (run_seq 1) with evidence
        ▼
   aggregate ④ any ERROR → FAILED · any WARN → WARNED (shadow) · else PASSED
        ▼
   route     ⑤ routing_policies first-match → file_transfers PENDING → atomic claim
        ▼
   move      ⑥ GET staging (re-hash verify) → PUT volume_map destination → verify → DELETE staging
              → transfer COMPLETED → file_registry ROUTED_* → lifecycle + events
```

**Inherited, non-negotiable rules** (Resilience Addendum; verified at R2 case 12):

* **Claim, don't poll-and-hope** — every stage transition is a guarded `findOneAndUpdate` on the expected current state; a crashed worker's claim expires and another picks the file up. No stage is ever "assumed".
* **Staging copy is deleted ONLY after** the destination PUT is HEAD-verified — worst case is a duplicated copy retried idempotently, never a lost file.
* **One FILE_ARRIVED → one routing outcome** — the `(file_id, direction) WHERE COMPLETED` partial-unique index makes double-delivery structurally impossible even if two workers race.
* **Poison containment** — a file that throws `FRAMEWORK_ERROR` on 3 consecutive attempts is auto-quarantined with `unidentified_reason: FRAMEWORK_ERROR_EXHAUSTED`; the pipeline keeps flowing.
* **Re-validate (run_seq ≥ 2)** uses this same engine with a different entry point — original evidence rows are never mutated.

**Process identity:** long-running async loop (one pod = one consumer), stateless between events, horizontal scale = more pods against the same `RECEIVED` backlog. All timing/metrics land at `/metrics` (`event_backlog`, per-stage latency, `staging_dir_count`).

## 2. The worker loop — claim, identify, validate, aggregate

```python
# worker/main.py (condensed; imports same as Step 3 + csv, json)
from datetime import datetime, timezone

IN_FLIGHT_MS = 5 * 60_000          # claim TTL — a crashed worker's event is re-drivable after this

async def run_worker(worker_id: str):
    while True:
        event = await claim_next_event(worker_id)      # ① atomic — see below
        if not event:
            await asyncio.sleep(poll_interval()); continue
        try:
            await process_file(event, worker_id)
            await mark_event(event, "PROCESSED")
        except FrameworkError as e:                     # infra/unexpected — retryable
            await handle_framework_error(event, e, worker_id)
        except QuarantineSignal:                         # poison-exhausted — contain
            pass                                        # handled inside; event still PROCESSED

async def claim_next_event(worker_id: str):
    """Competing consumers: exactly one worker wins each event (guarded update)."""
    # review V1: an UNCLAIMED event has no claimed_at field — a bare $lt never matches it.
    # The $or lets fresh events claim instantly and stale PROCESSING claims re-drive after TTL.
    cutoff = datetime.now(timezone.utc) - timedelta(milliseconds=IN_FLIGHT_MS)
    claim_filter = {"event_type": "FILE_ARRIVED", "status": "RECEIVED",
                     "$or": [{"claimed_at": {"$exists": False}},
                             {"claimed_at": {"$lt": cutoff}}]}
    return await mongo.db.event_log.find_one_and_update(
        claim_filter,
        {"$set": {"status": "PROCESSING", "claimed_at": datetime.now(timezone.utc),
                  "worker_id": worker_id}},
        sort=[("timestamp", 1)])                        # review V5: (key, direction) — FIFO on backlog

async def process_file(event, worker_id: str):
    fr = await mongo.db.file_registry.find_one_and_update(   # state guard: RECEIVED -> IDENTIFYING
        {"file_id": event["dedup_key"], "current_status": "RECEIVED"},
        {"$set": {"current_status": "IDENTIFYING", "updated_at": datetime.now(timezone.utc)}})
    if not fr:
        return                                          # already claimed by a prior crash-retry

    # ② IDENTIFY — registry-driven, no hard-coded types (explicit > filename > header)
    ident = await identify(fr)                           # -> {file_type_id, method, confidence, evidence}
    if not ident:
        await finalize_unidentified(fr); return          # -> UNIDENTIFIED + quarantine via policy (R2 case 11)
    await mongo.db.file_registry.update_one(
        {"file_id": fr["file_id"], "current_status": "IDENTIFYING"},
        {"$set": {"current_status": "VALIDATING", "file_type_id": ident["file_type_id"],
                  "identification_method": ident["method"],
                  "identification_confidence": ident["confidence"],
                  "identification_evidence": ident["evidence"],
                  "schema_version_id": ident["schema_version_id"],
                  "updated_at": datetime.now(timezone.utc)}})

    # ③ VALIDATE — rules-as-data: pull active rules for this type, run, write evidence
    rules = await load_rules(ident["file_type_id"])       # validation_rules, priority-ordered
    schema = await load_active_schema(ident["file_type_id"])  # schema_versions (the real contracts)
    sf = StagedFile(fr, dbx.client)                      # review V3: ONE cached download per run
    verdicts = []
    for rule in rules:                                   # FILE category re-verified cheaply; rest here
        v = RULE_PLUGINS[rule["rule_code"]](fr, rule, schema, sf)
        verdicts.append(v)                               # {rule_id, rule_version, status, details, duration_ms}
        await mongo.db.validation_results.insert_one({    # run_seq 1 evidence — immutable trail
            "result_id": "vr_" + uuid.uuid4().hex[:16], "file_id": fr["file_id"],
            "batch_id": None, "run_seq": 1, "rule_id": rule["rule_id"],
            "rule_version": rule["version"], "status": v["status"],
            "details": v["details"], "executed_at": datetime.now(timezone.utc),
            "duration_ms": v["duration_ms"]})

    # ④ AGGREGATE — ERROR dominates, WARN shadows at launch (R1)
    statuses = {v["status"] for v in verdicts}
    new_status = "FAILED" if "FAIL" in statuses else "WARNED" if "WARN" in statuses else "PASSED"
    await mongo.db.file_registry.update_one(
        {"file_id": fr["file_id"], "current_status": "VALIDATING"},   # guarded transition
        {"$set": {"current_status": new_status, "updated_at": datetime.now(timezone.utc)}})
    await emit({"event_type": "VALIDATION_COMPLETED", "dedup_key": f"{fr['file_id']}:1",
                "payload": {"file_id": fr["file_id"], "status": new_status}})
    await route_and_move(fr, new_status)                 # -> cell 3
```

> `FRAMEWORK_ERROR` handling (`handle_framework_error`): increment `retry_count` (guarded), leave the event `RECEIVED` so it re-drives, and after 3 exhausted attempts flip the file to `ROUTED_QUARANTINE` via the transfer path with `unidentified_reason: FRAMEWORK_ERROR_EXHAUSTED` + `FILE_STAGED` ops notification. Event is marked `DEAD` only if it exceeds a 24h age — the janitor sweeps `DEAD` for review.

## 3. `route_and_move` — policies → transfer claim → the byte move

```python
async def route_and_move(fr, new_status: str):
    # ⑤ ROUTE — first matching policy by priority; every seed file matches exactly one
    policies = await mongo.db.routing_policies.find(
        {"is_active": True}).sort("priority", 1).to_list(None)
    pol = next((p for p in policies if policy_matches(p, new_status, fr)), None)
    action = pol["action"] if pol else "MOVE_QUARANTINE"      # fail-safe: no match -> quarantine

    direction = "MONGO_TO_QUARANTINE" if action == "MOVE_QUARANTINE" else "MONGO_TO_INTAKE"
    dest_status = "ROUTED_QUARANTINE" if action == "MOVE_QUARANTINE" else "ROUTED_INTAKE"

    tid = "trf_" + uuid.uuid4().hex[:16]
    await mongo.db.file_transfers.insert_one({       # the manifest of the move (C4 guard target)
        "transfer_id": tid, "file_id": fr["file_id"], "policy_id": pol and pol["policy_id"],
        "from_location": f"volume:{fr['staged_path']}", "to_location": "",
        "direction": direction, "status": "PENDING", "attempts": 0,
        "last_error": None, "source_sha256": fr["sha256_checksum"],
        "dest_sha256": None, "bytes_written": None, "worker_id": None,
        "started_at": None, "completed_at": None})
    t = await mongo.db.file_transfers.find_one_and_update(    # atomic claim — one worker moves it
        {"transfer_id": tid, "status": "PENDING"},
        {"$set": {"status": "IN_PROGRESS",              # review V2: $inc is a SIBLING operator,
                  "worker_id": WORKER_ID, "started_at": datetime.now(timezone.utc)},
         "$inc": {"attempts": 1}})                       # nesting it in $set is a runtime error
    if not t:
        return                                            # a retrying worker already owns it

    # ⑥ MOVE — staging volume -> intake|quarantine (volume_map rows, real roots)
    if fr["size_bytes"] > 64 * 1024 * 1024:                # review V6: whole-body read guard until
        raise FrameworkError("file >64MB — chunked move not built (Storage Addendum threshold)")
    vm = await resolve_volume_map(fr, action)              # INTAKE row by file_type_id / QUARANTINE catch-all
    r = await dbx.client.get_file(fr["staged_path"])        # download (small file profile)
    r.raise_for_status()
    body = r.content
    sha = hashlib.sha256(body).hexdigest()
    if sha != fr["sha256_checksum"]:
        raise FrameworkError("staged bytes hash mismatch vs registry — quarantine path, investigate")

    dest_path = render_destination(vm, fr)                 # <root>/<subfolder>/{file_type}_{file_id}_{ts}.csv
    pr = await dbx.client.put_file(dest_path, body)        # overwrite=true — idempotent re-drive
    pr.raise_for_status()
    hv = await dbx.client.head(dest_path)                  # verify BEFORE deleting staging
    if hv.status_code != 200:
        raise FrameworkError(f"dest not verifiable after PUT: {dest_path}")

    await dbx.client.delete_file(fr["staged_path"])          # staging cleanup — only now
    async with await mongo.client.start_session() as ms:    # commit the outcome atomically
        async with ms.start_transaction():
            await mongo.db.file_transfers.update_one(ms,
                {"transfer_id": tid},
                {"$set": {"status": "COMPLETED", "to_location": f"volume:{dest_path}",
                          "dest_sha256": sha, "bytes_written": len(body),
                          "completed_at": datetime.now(timezone.utc)}})   # partial-unique guard fires here on races
            await mongo.db.file_registry.update_one(ms,
                {"file_id": fr["file_id"], "current_status": new_status},
                {"$set": {"current_status": dest_status,
                          "updated_at": datetime.now(timezone.utc)}})
            await mongo.db.file_lifecycle.insert_one(ms, {
                "event_id": "flc_" + uuid.uuid4().hex[:16], "file_id": fr["file_id"],
                "location": "INTAKE_VOLUME" if action == "MOVE_INTAKE" else "QUARANTINE",
                "timestamp": datetime.now(timezone.utc), "triggered_by_job": None})
            await emit({"event_type": "TRANSFER_COMPLETED",       # review V4: canonical emit
                        "dedup_key": f"{tid}:done",
                        "payload": {"file_id": fr["file_id"], "to": dest_path}}, ms=ms)
            await emit({"event_type": "FILE_ROUTED", "dedup_key": f"{fr['file_id']}:routed",
                        "payload": {"file_id": fr["file_id"], "status": dest_status}}, ms=ms)
    await audit("worker", "file_registry", fr["file_id"], "FILE_ROUTED", {},
                {"status": dest_status, "destination": dest_path})
```

**Move-failure ladder** (`TRANSFER_FAILED` event + notification `not_transfer_failed`): transfer stays `RETRYING` with backoff (re-drive from `PENDING→RETRYING` scan), attempts capped at 3, then the file is flagged for ops on the board — **the staging copy is never deleted while a transfer is unfinished**, so nothing is ever lost mid-move; a re-drive re-PUTs the destination idempotently (overwrite=true).

## 4. Rule plugins — rules-as-data, eight executors

```python
class StagedFile:                      # review V3: ONE download per file per run — lazy, cached
    """Plugins read the staged file through this wrapper; bytes are fetched from the
    staging volume on first use and memoized for the whole rule run (the naive
    per-plugin get_file re-downloaded the file 3–4× per validation)."""
    def __init__(self, fr, files_client):
        self._fr, self._fc, self._lines = fr, files_client, None
    def lines(self) -> list[str]:                      # decoded, split, memoized
        if self._lines is None:
            body = run_sync(self._fc.get_file, self._fr["staged_path"]).content
            self._lines = body.decode(self._fr.get("encoding") or "utf-8").splitlines()
        return self._lines

RULE_PLUGINS = {}                      # rule_code -> (fr, rule, schema, sf) -> verdict

def plugin(code):
    def deco(fn):
        RULE_PLUGINS[code] = fn; return fn
    return deco

@plugin("HEADER_REQUIRED")
def header_required(fr, rule, schema, sf):                     # review V3: reads via cached sf
    lines = sf.lines()
    ok = bool(lines and lines[0].strip())
    return {"status": "PASS" if ok else "FAIL",
            "details": {"header_row_found": ok}, "duration_ms": 0}

@plugin("SCHEMA_MATCH")
def schema_match(fr, rule, schema, sf):
    expected = [f["name"] for f in schema["fields"]]          # the real mdd_platform-derived contracts
    lines = sf.lines()
    header = [c.strip() for c in lines[0].split(",")] if lines else []
    missing, extra = set(expected) - set(header), set(header) - set(expected)
    ok = not missing and not extra
    return {"status": "PASS" if ok else "FAIL",
            "details": {"expected": expected, "actual": header,
                        "missing": sorted(missing), "extra": sorted(extra)}, "duration_ms": 0}

@plugin("COLUMN_ORDER_MATCH")
def column_order(fr, rule, schema, sf):
    expected = [f["name"] for f in schema["fields"]]
    lines = sf.lines()
    header = [c.strip() for c in lines[0].split(",")] if lines else []
    ok = header == expected
    return {"status": "PASS" if ok else "FAIL",
            "details": {"expected_order": expected, "actual_order": header}, "duration_ms": 0}

@plugin("FILE_NOT_EMPTY_ROWS")
def not_empty_rows(fr, rule, schema, sf):                        # WARN — shadows at launch (R1)
    rows = sf.lines()[1:]
    ok = len(rows) >= rule["params"]["min_rows"]
    return {"status": "PASS" if ok else "WARN",
            "details": {"rows": len(rows), "min_rows": rule["params"]["min_rows"]}, "duration_ms": 0}

# FILE_TYPE_ALLOWED / FILE_SIZE_MAX / FILE_NOT_EMPTY / FILE_NOT_DUPLICATE:
#   re-verified here against the registry row itself (cheap, no volume read) —
#   the API already enforced them pre-staging; the worker re-checks to keep
#   evidence complete per run (one PASS row each). Unknown rule_code -> SKIPPED + ops alert.
```

> Adding a rule at any time = insert a `validation_rules` document + register a plugin. Nothing else changes — the engine, evidence, aggregation, routing and audit are all registry-driven. (Re-validate feeds the same plugins with `run_seq = max+1`.)

## 5. Verification — worker smoke against the real volumes (runnable here)

The worker itself runs in the service deployment, but its **byte-move mechanics against the real `mdd_platform.fvr` volumes** can be proven from Databricks compute now — the same PUT → verify → delete-staging sequence the worker executes:

```python
from pathlib import Path
import hashlib, shutil, datetime

base = Path("/Volumes/mdd_platform/fvr")
sess, name = base / "staging" / "sessions" / "ses_worker_smoke", "orders_smoke.csv"
sess.mkdir(parents=True, exist_ok=True)
staged = sess / name
body = "order_id,customer_id,order_date,amount,currency,status,channel\nORD-1,C-1,2026-11-09,99.5,USD,shipped,portal\n".encode()
staged.write_bytes(body)                                   # pretend the API staged it
sha = hashlib.sha256(body).hexdigest()

dest = base / "intake" / "order" / f"ft_order_file_smoke_{datetime.datetime.utcnow():%Y%m%d%H%M%S}.csv"
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_bytes(body)                                     # the move (PUT)
assert dest.stat().st_size == len(body)                     # HEAD-verify BEFORE deleting staging
assert hashlib.sha256(dest.read_bytes()).hexdigest() == sha # dest hash == registry sha256
staged.unlink(); sess.rmdir()                               # staging cleanup — only now
print("move OK →", dest, "| sha verified | staging clean")
dest.unlink()                                              # smoke cleanup
```

**Step-4 pass criteria:**

1. The smoke above passes on the real volumes.
2. Deployed against dev Mongo: feed one valid order file through Step 3's API, watch the worker take it `RECEIVED → ROUTED_INTAKE` with 8 `validation_results` PASS rows, `file_lifecycle` showing `STAGING_VOLUME → INTAKE_VOLUME`, and the staging directory gone.
3. Kill the worker mid-validation (R2 case 12) — after the claim TTL the event re-drives; no duplicate `validation_results`, no double transfer (the partial-unique index is the backstop).

## 6. Build status after Step 4

| Component | Status |
| --- | --- |
| Registry + operational collections (Steps 1–2) | ✅ DDL + seeds authored (roots real) |
| Upload API (Step 3) | ✅ authored + reviewed (F1–F4 fixed) |
| Validation & routing worker (Step 4) | ✅ this notebook |
| Test fixtures for R2 | `fvr_test_files/` in the repo (generated with the R2 notebook) |
| **R2 execution** | ← next: run the 12-case matrix + locust S1–S5, record the verdict |

Remaining Phase-1 build items after R2: OPS read APIs (board, quarantine re-classify/re-validate endpoints), the janitor job (abandoned-session directories + `DEAD` events), and the React portal.

## 6. Code review (2026-11-09) — findings & fixes applied

| # | Severity | Finding | Fix applied |
| --- | --- | --- | --- |
| V1 | **blocker** | claim filter used `claimed_at: {$lt: TTL}` — but a **never-claimed** event has no `claimed_at` field, so `$lt` never matches it: **every fresh event would be invisible**; only TTL-expired re-drives would ever claim | `$or: [{$exists: false}, {$lt: cutoff}]` — fresh events claim instantly, stale PROCESSING claims re-drive after the 5-min TTL |
| V2 | **blocker** | transfer claim used `$set: {attempts: {$inc: 1}}` — `$inc` nested inside `$set` is invalid update syntax → `WriteError` on every first move | `$inc` moved to a sibling operator: `{$set: {...}, $inc: {attempts: 1}}` |
| V3 | major | each HEADER/CONTENT plugin did its own `get_file` — the same file downloaded 3–4× per validation run (the "cached in-process" comment described intent, not behavior) | `StagedFile` lazy-cached reader: one download per file per run; all four plugins now take `sf` and read `sf.lines()`; plugin calls are plain (no `await`) |
| V4 | major | `emit()` called with two different signatures (`emit(ms, {...})` in transactions, `emit({...})` outside); helpers `first_header_row`/`data_rows` referenced but never defined | canonical `emit(payload, ms=None)` — joins the caller's session when inside a transaction, opens its own otherwise; helpers eliminated by V3's `sf.lines()` |
| V5 | minor | `sort=["timestamp"]` — motor wants `(key, direction)` tuples | `sort=[("timestamp", 1)]` |
| V6 | minor | `route_and_move` reads the whole body into memory with no ceiling — a future 500 MB file would balloon worker memory before the size rule caught it (the API rejects >500 MB at manifest, but defense-in-depth belongs here too) | explicit `>64MB → FrameworkError` guard until chunked sessions exist (aligned with the Storage Addendum ~100 MB threshold) |

**Helper contracts (defined once, in `worker/support.py` — signatures fixed by this review):** `emit(payload: dict, ms=None)`; `audit(actor, entity_type, entity_id, action, before, after)`; `identify(fr) -> dict | None`; `finalize_unidentified(fr)` → emits `FILE_UNIDENTIFIED` and **calls `route_and_move(fr, "UNIDENTIFIED")`** so policies 20/30 decide the quarantine move (never hard-coded); `load_rules(ft_id)`; `load_active_schema(ft_id)`; `resolve_volume_map(fr, action)`; `render_destination(vm, fr)`; `run_sync(coro_fn, *args)` (asyncio.run_coroutine_threadsafe against the shared loop); `class FrameworkError(Exception)`; `class QuarantineSignal(Exception)` raised by `handle_framework_error` after 3 exhausted attempts.

**Verdict:** V1/V2 were latent runtime failures (claim starvation / write error on first move) — both fixed; with V3/V4 the hot path does one volume read per file and one consistent event API. Sound for dev deployment alongside the reviewed Step 3.